In [2]:
%pip install datasets nltk

  Using cached click-8.4.2-py3-none-any.whl.metadata (2.6 kB)
   ---------------------------------------- 0.0/559.1 kB ? eta -:--:--
   ------------------ --------------------- 262.1/559.1 kB ? eta -:--:--
   ---------------------------------------- 559.1/559.1 kB 1.8 MB/s  0:00:00
   ---------------------------------------- 0.0/784.9 kB ? eta -:--:--
   -------------------------- ------------- 524.3/784.9 kB 3.2 MB/s eta 0:00:01
   ---------------------------------------- 784.9/784.9 kB 2.8 MB/s  0:00:00
Using cached click-8.4.2-py3-none-any.whl (119 kB)
   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   -- ------------------------------------- 0.3/4.0 MB ? eta -:--:--
   ----- ---------------------------------- 0.5/4.0 MB 2.8 MB/s eta 0:00:02
   ------------ --------------------------- 1.3/4.0 MB 2.3 MB/s eta 0:00:02
   ------------------ --------------------- 1.8/4.0 MB 2.4 MB/s eta 0:00:01
   -------------------- ------------------- 2.1/4.0 MB 2.3 MB/s eta 0:00

In [1]:
# ============================================================
# SMART NEXT-WORD PREDICTOR
# Using WikiText-2 + Unigram + Bigram + Trigram
# ============================================================

import re
import urllib.request
from collections import Counter, defaultdict


# ============================================================
# 1. LOAD WIKITEXT-2 CORPUS
# ============================================================

print("Loading WikiText-2 corpus...")

url = "https://raw.githubusercontent.com/pytorch/examples/main/word_language_model/data/wikitext-2/train.txt"

try:
    with urllib.request.urlopen(url) as response:
        train_text = response.read().decode("utf-8")

    print("Corpus loaded successfully!")

except Exception as e:
    print("Error loading WikiText-2:", e)


# ============================================================
# 2. CLEAN AND TOKENIZE TEXT
# ============================================================

def tokenize(text):
    """
    Convert text into lowercase word tokens.
    """

    # Convert to lowercase
    text = text.lower()

    # Remove unwanted characters
    text = re.sub(r"[^a-z0-9\s']", " ", text)

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    # Split into words
    tokens = text.split()

    return tokens


tokens = tokenize(train_text)

print("Total tokens:", len(tokens))


# ============================================================
# 3. BUILD UNIGRAM FREQUENCY TABLE
# ============================================================

unigram_counts = Counter(tokens)

print("Unigram table created.")


# ============================================================
# 4. BUILD BIGRAM FREQUENCY TABLE
# ============================================================

bigram_counts = Counter(
    zip(tokens[:-1], tokens[1:])
)

print("Bigram table created.")


# ============================================================
# 5. BUILD TRIGRAM FREQUENCY TABLE
# ============================================================

trigram_counts = Counter(
    zip(
        tokens[:-2],
        tokens[1:-1],
        tokens[2:]
    )
)

print("Trigram table created.")


# ============================================================
# 6. CREATE BIGRAM NEXT-WORD TABLE
# ============================================================

bigram_next_words = defaultdict(Counter)

for (word1, word2), count in bigram_counts.items():
    bigram_next_words[word1][word2] = count


# ============================================================
# 7. CREATE TRIGRAM NEXT-WORD TABLE
# ============================================================

trigram_next_words = defaultdict(Counter)

for (word1, word2, word3), count in trigram_counts.items():
    trigram_next_words[(word1, word2)][word3] = count


print("N-gram tables created successfully!")


# ============================================================
# 8. BIGRAM PROBABILITY
# ============================================================

def get_bigram_predictions(previous_word, top_n=5):

    candidates = bigram_next_words.get(
        previous_word,
        {}
    )

    if not candidates:
        return []

    total = sum(candidates.values())

    predictions = []

    for word, count in candidates.items():

        probability = count / total

        predictions.append(
            (word, probability)
        )

    # Sort by probability
    predictions.sort(
        key=lambda x: x[1],
        reverse=True
    )

    return predictions[:top_n]


# ============================================================
# 9. TRIGRAM PROBABILITY
# ============================================================

def get_trigram_predictions(
    previous_word1,
    previous_word2,
    top_n=5
):

    candidates = trigram_next_words.get(
        (previous_word1, previous_word2),
        {}
    )

    if not candidates:
        return []

    total = sum(candidates.values())

    predictions = []

    for word, count in candidates.items():

        probability = count / total

        predictions.append(
            (word, probability)
        )

    # Sort by probability
    predictions.sort(
        key=lambda x: x[1],
        reverse=True
    )

    return predictions[:top_n]


# ============================================================
# 10. UNIGRAM PROBABILITY
# ============================================================

def get_unigram_predictions(top_n=5):

    total = sum(unigram_counts.values())

    predictions = []

    for word, count in unigram_counts.most_common(top_n):

        probability = count / total

        predictions.append(
            (word, probability)
        )

    return predictions


# ============================================================
# 11. NEXT-WORD PREDICTOR
# ============================================================

def predict_next_words(sentence, top_n=5):

    # Tokenize input sentence
    words = tokenize(sentence)

    # If input is empty
    if not words:
        return [], "None"

    # --------------------------------------------------------
    # First try TRIGRAM
    # --------------------------------------------------------

    if len(words) >= 2:

        word1 = words[-2]
        word2 = words[-1]

        predictions = get_trigram_predictions(
            word1,
            word2,
            top_n
        )

        if predictions:
            return predictions, "Trigram"


    # --------------------------------------------------------
    # If trigram unavailable, use BIGRAM
    # --------------------------------------------------------

    last_word = words[-1]

    predictions = get_bigram_predictions(
        last_word,
        top_n
    )

    if predictions:
        return predictions, "Bigram"


    # --------------------------------------------------------
    # If bigram unavailable, use UNIGRAM
    # --------------------------------------------------------

    predictions = get_unigram_predictions(top_n)

    return predictions, "Unigram"


# ============================================================
# 12. DISPLAY PREDICTIONS
# ============================================================

def display_predictions(sentence, top_n=5):

    predictions, method = predict_next_words(
        sentence,
        top_n
    )

    print("\n" + "=" * 55)
    print("SMART NEXT-WORD PREDICTOR")
    print("=" * 55)

    print("\nInput:")
    print(sentence)

    print("\nPrediction Method:")
    print(method)

    print("\nTop predicted next words:")

    if not predictions:

        print("No predictions found.")

        return

    for i, (word, probability) in enumerate(
        predictions,
        start=1
    ):

        print(
            f"{i}. {word:<15} - "
            f"{probability:.3f} "
            f"({probability * 100:.2f}%)"
        )

    print("=" * 55)


# ============================================================
# 13. TEST THE PREDICTOR
# ============================================================

print("\n")
print("=" * 55)
print("WikiText-2 Smart Next-Word Predictor is Ready!")
print("=" * 55)

print("\nExample test:")

display_predictions(
    "Machine learning is",
    top_n=5
)

Loading WikiText-2 corpus...
Corpus loaded successfully!
Total tokens: 1755457
Unigram table created.
Bigram table created.
Trigram table created.
N-gram tables created successfully!


WikiText-2 Smart Next-Word Predictor is Ready!

Example test:

SMART NEXT-WORD PREDICTOR

Input:
Machine learning is

Prediction Method:
Bigram

Top predicted next words:
1. a               - 0.122 (12.15%)
2. the             - 0.079 (7.85%)
3. not             - 0.030 (2.98%)
4. also            - 0.026 (2.57%)
5. an              - 0.023 (2.26%)
